# 아이 키 성장 기록 × 2017 성장도표 비교 분석

## 분석 질문
1. 아이의 키는 또래 남아 중앙값과 비교해 cm 차이와 백분위 기준으로 각각 어떻게 변해 왔는가?
2. 아이의 키 백분위가 가장 크게 오른 시기는 언제인가?
3. 4계절(봄·여름·가을·겨울)에 따라 아이의 성장 속도에 차이가 있는가?
4. 최근 1년 평균 백분위가 유지된다면 18세 때 키는 얼마일까?

## 3단계. 데이터 로드 및 기본 정보 확인

로컬에서 실행하면 `data/` 폴더를, Colab에서 실행하면 GitHub 저장소의 파일을 읽는다.

In [ ]:
from pathlib import Path
import pandas as pd

if Path("data").exists():
    BASE = "data/"
else:
    BASE = "https://raw.githubusercontent.com/Hoyasiro/M1-1/main/data/"

child = pd.read_csv(BASE + "child_growth_anon.csv")
child.shape

### 3-1. 컬럼과 자료형

In [ ]:
child.info()

### 3-2. 앞부분 미리보기

In [ ]:
child.head(15)

### 3-3. 컬럼별 결측치 개수

In [ ]:
child.isna().sum()

## 4단계. 결측치·이상치 처리

### 4-1. 결측치 처리
- 모든 컬럼이 빈 5행: 원본 엑셀에서 보기 편하도록 끼워 넣은 구분용 빈 줄이므로 삭제
- 키만 빈 행: 원본은 육아 기록이라 키 측정이 그날의 주목적이 아니었음. 재지 않은 값을 만들지 않기 위해 키 분석에서는 제외(실측 66개만 사용)

In [ ]:
child = child.dropna(how="all")
height = child.dropna(subset=["키"]).copy()
print(child.shape, height.shape)

### 4-2. 시간 순서 확인 (`days`가 계속 커지는가?)

In [ ]:
height["days"].is_monotonic_increasing

### 4-3. 같은 날 중복 측정

In [ ]:
height[height.duplicated("days", keep=False)]

### 4-4. 이전 측정보다 키가 줄어든 경우

In [ ]:
height = height.sort_values("days")
height["키_변화"] = height["키"].diff()
height[height["키_변화"] < 0]

### 4-5. 키 감소 이상치 처리
- 기준: 이전 측정보다 0.8cm 넘게 줄어든 값은 삭제
- 이유: 0.5~0.6cm 감소는 측정 자세·시간대에 따른 일반적 오차 범위로 보고 유지(이동평균으로 완화 가능).
  1.2cm 감소는 그보다 훨씬 커서 남겨 두면 이동평균선까지 끌어내려 추세를 왜곡할 수 있음

In [ ]:
height = height[~(height["키_변화"] < -0.8)].copy()
height["키_변화"] = height["키"].diff()
height.shape

### 4-6. 짧은 기간에 크게 뛴 값
측정 간격이 제각각이므로 키 변화를 1년 기준 속도(cm/년)로 바꿔 비교한다.

In [ ]:
height["간격_일"] = height["days"].diff()
height["성장속도_cm_년"] = height["키_변화"] / height["간격_일"] * 365
height.sort_values("성장속도_cm_년", ascending=False).head(5)